# Week 12 Mini Assignment — PySpark Data Processing Pipeline
**Dataset:** NYC TLC Yellow Taxi trip records, Jan 2023 – Dec 2024 (24 Parquet files, ~1.2 GB compressed, ~78M rows)
+ Taxi Zone lookup table (265 zones) for joins.

Sections
1. Setup & loading
2. Actions vs. transformations (lazy evaluation demo)
3. Transformation pipeline (early filters, `withColumn`, broadcast join, `groupBy`)
4. Spark SQL queries
5. Execution plans (`.explain()`) & optimization
6. Caching benchmark
7. Writing results (partitioned Parquet)
8. MLlib (optional) — fare regression

## 1. Setup & loading

In [1]:
import time, glob, os
from functools import reduce
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import DoubleType, IntegerType, LongType

os.makedirs("spark-events", exist_ok=True)
spark = (SparkSession.builder
         .config("spark.eventLog.enabled", "true")          # lets the Spark History Server replay the UI later
         .config("spark.eventLog.dir", "spark-events")
         .appName("NYC-Taxi-PySpark-Pipeline")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "64")      # default 200 is too many for one machine
         .config("spark.sql.adaptive.enabled", "true")       # AQE: coalesce shuffle partitions, skew joins
         .config("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

Spark 3.5.3 | UI: http://macbookpro.lan:4040


In [2]:
DATA = "data"
files = sorted(glob.glob(f"{DATA}/yellow/yellow_tripdata_*.parquet"))
total_gb = sum(os.path.getsize(f) for f in files) / 1024**3
print(f"{len(files)} files, {total_gb:.2f} GB on disk")

# The monthly TLC files are not perfectly schema-consistent (e.g. INT32 vs INT64, airport_fee casing),
# so each file is read with an explicit projection + cast and the results are unioned by name.
# Selecting only the needed columns here is also our column-pruning step: Parquet is columnar,
# so the other columns are never read from disk.
COLS = {
    "tpep_pickup_datetime": None, "tpep_dropoff_datetime": None,
    "passenger_count": IntegerType(), "trip_distance": DoubleType(),
    "PULocationID": IntegerType(), "DOLocationID": IntegerType(),
    "payment_type": IntegerType(), "fare_amount": DoubleType(),
    "tip_amount": DoubleType(), "total_amount": DoubleType(),
}
def load(path):
    df = spark.read.parquet(path)
    return df.select([F.col(c).cast(t).alias(c) if t else F.col(c) for c, t in COLS.items()])

trips_raw = reduce(lambda a, b: a.unionByName(b), [load(f) for f in files])

zones = (spark.read.option("header", True).option("inferSchema", True)
         .csv(f"{DATA}/taxi_zone_lookup.csv"))
trips_raw.printSchema()
zones.show(5)

24 files, 1.24 GB on disk


root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+----

In [3]:
t = time.time()
n_raw = trips_raw.count()
print(f"raw rows: {n_raw:,}  ({time.time()-t:.1f}s)")

raw rows: 79,479,946  (3.2s)


## 2. Actions vs. Transformations
Transformations (`filter`, `withColumn`, `select`, `groupBy`, `join`) only build a logical plan and return instantly.
Actions (`count`, `show`, `collect`, `write`) trigger a Spark job that actually reads and processes data.

In [4]:
t0 = time.time()
lazy_df = (trips_raw
           .filter(F.col("trip_distance") > 10)
           .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
           .groupBy("PULocationID").agg(F.avg("fare_per_mile").alias("avg_fpm")))
t_transform = time.time() - t0
print(f"Transformations defined in {t_transform*1000:.1f} ms — no job ran (check Spark UI: no new job)")
print(type(lazy_df))

t0 = time.time()
result = lazy_df.orderBy(F.desc("avg_fpm")).limit(5).collect()   # ACTION
t_action = time.time() - t0
print(f"Action (collect) took {t_action:.2f} s — this is when Spark actually read the Parquet files")
for r in result: print(r)

Transformations defined in 87.7 ms — no job ran (check Spark UI: no new job)
<class 'pyspark.sql.dataframe.DataFrame'>


Action (collect) took 10.58 s — this is when Spark actually read the Parquet files
Row(PULocationID=157, avg_fpm=5.098134897625774)
Row(PULocationID=226, avg_fpm=4.904244012435546)
Row(PULocationID=207, avg_fpm=4.894900433248408)
Row(PULocationID=160, avg_fpm=4.839343707976048)
Row(PULocationID=1, avg_fpm=4.824407499115296)


## 3. Transformation pipeline
Filters are applied **first** (right after the projection) so that every later step — derived columns,
the join and the shuffle for `groupBy` — operates on fewer rows. Simple comparison filters on Parquet
columns are pushed into the scan (`PushedFilters` in the plan).

In [5]:
START, END = "2023-01-01", "2025-01-01"

trips = (trips_raw
    # --- filter 1: valid time window (files contain a few mis-dated records) ---
    .filter((F.col("tpep_pickup_datetime") >= START) & (F.col("tpep_pickup_datetime") < END))
    # --- filter 2: physically plausible trips ---
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    # --- filter 3: valid money / passengers ---
    .filter((F.col("fare_amount") > 0) & (F.col("fare_amount") < 500) & (F.col("passenger_count") > 0))
    # --- column transformations ---
    .withColumn("duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("year_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    .withColumn("tip_pct", F.when(F.col("fare_amount") > 0, F.col("tip_amount") / F.col("fare_amount") * 100))
    .withColumn("payment", F.when(F.col("payment_type") == 1, "Credit card")
                             .when(F.col("payment_type") == 2, "Cash")
                             .otherwise("Other"))
    # filter 4 depends on a derived column, so it must come after withColumn
    .filter((F.col("duration_min") > 1) & (F.col("duration_min") < 180))
    .withColumn("speed_mph", F.col("trip_distance") / (F.col("duration_min") / 60.0))
)

# --- join: zones is tiny (265 rows) -> broadcast join, no shuffle of the 78M-row side ---
pu = zones.select(F.col("LocationID").alias("PULocationID"),
                  F.col("Borough").alias("pu_borough"), F.col("Zone").alias("pu_zone"))
do = zones.select(F.col("LocationID").alias("DOLocationID"),
                  F.col("Borough").alias("do_borough"), F.col("Zone").alias("do_zone"))
enriched = (trips
            .join(F.broadcast(pu), "PULocationID", "left")
            .join(F.broadcast(do), "DOLocationID", "left"))
enriched.select("tpep_pickup_datetime", "pu_zone", "do_zone", "trip_distance",
                "duration_min", "speed_mph", "tip_pct", "payment").show(5, truncate=False)

+--------------------+--------------+---------------------+-------------+------------------+------------------+------------------+-----------+
|tpep_pickup_datetime|pu_zone       |do_zone              |trip_distance|duration_min      |speed_mph         |tip_pct           |payment    |
+--------------------+--------------+---------------------+-------------+------------------+------------------+------------------+-----------+
|2023-01-01 00:32:10 |Midtown Center|Lenox Hill West      |0.97         |8.433333333333334 |6.901185770750987 |0.0               |Cash       |
|2023-01-01 00:55:08 |Central Park  |Upper East Side South|1.1          |6.316666666666666 |10.448548812664908|50.632911392405056|Credit card|
|2023-01-01 00:25:04 |Clinton East  |Upper West Side North|2.51         |12.75             |11.811764705882352|100.67114093959731|Credit card|
|2023-01-01 00:10:29 |Gramercy      |East Village         |1.43         |10.833333333333334|7.92              |28.771929824561397|Credit card|

In [6]:
# --- groupBy with multiple aggregations ---
t = time.time()
borough_hour = (enriched
    .groupBy("pu_borough", "pickup_hour")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("trip_distance"), 2).alias("avg_miles"),
         F.round(F.expr("percentile_approx(duration_min, 0.5)"), 1).alias("median_min"),
         F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct"),
         F.round(F.sum("total_amount") / 1e6, 2).alias("revenue_musd"))
    .orderBy("pu_borough", "pickup_hour"))
borough_hour.show(24)
print(f"{time.time()-t:.1f}s")

+----------+-----------+-----+--------+---------+----------+-----------+------------+
|pu_borough|pickup_hour|trips|avg_fare|avg_miles|median_min|avg_tip_pct|revenue_musd|
+----------+-----------+-----+--------+---------+----------+-----------+------------+
|     Bronx|          0|  865|   28.58|     5.84|      14.2|        8.1|        0.03|
|     Bronx|          1|  596|   23.12|     4.66|      11.4|       7.31|        0.02|
|     Bronx|          2|  423|   24.67|     4.81|      11.4|       5.61|        0.01|
|     Bronx|          3|  482|   31.53|     7.36|      16.6|       5.96|        0.02|
|     Bronx|          4| 2308|   38.47|     9.83|      32.4|       1.63|         0.1|
|     Bronx|          5| 5039|   39.95|    10.51|      36.3|       0.37|        0.22|
|     Bronx|          6| 8667|   37.18|     9.57|      42.0|       0.26|        0.35|
|     Bronx|          7| 8516|   33.53|      8.1|      41.2|       0.82|        0.31|
|     Bronx|          8| 9972|   33.32|     7.97|     

In [7]:
monthly = (enriched.groupBy("year_month")
           .agg(F.count("*").alias("trips"),
                F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"))
           .orderBy("year_month"))
monthly.show(24)

+----------+-------+--------+-------------+
|year_month|  trips|avg_fare|avg_speed_mph|
+----------+-------+--------+-------------+
|   2023-01|2872728|   18.51|        12.33|
|   2023-02|2721723|   18.41|        11.86|
|   2023-03|3177255|    19.1|        11.64|
|   2023-04|3064566|   19.55|        11.73|
|   2023-05|3268104|   20.05|        11.25|
|   2023-06|3072504|   20.02|        11.42|
|   2023-07|2701475|   19.94|        11.92|
|   2023-08|2618158|   19.97|        11.93|
|   2023-09|2594386|   20.71|        10.81|
|   2023-10|3231046|   20.28|        10.75|
|   2023-11|3079670|   19.89|        10.64|
|   2023-12|3063795|   19.85|        10.56|
|   2024-01|2713298|   18.39|         11.5|
|   2024-02|2709585|   18.33|        11.13|
|   2024-03|3024172|   19.11|        11.16|
|   2024-04|2976046|    19.4|        11.04|
|   2024-05|3180269|   19.94|        10.61|
|   2024-06|2994458|   19.76|        10.87|
|   2024-07|2671430|   20.15|         11.4|
|   2024-08|2591962|   20.36|   

## 4. Spark SQL queries

In [8]:
enriched.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

# SQL 1: top 10 pickup zones by revenue, with share of total
top_zones = spark.sql('''
    SELECT pu_borough, pu_zone,
           COUNT(*)                         AS trips,
           ROUND(SUM(total_amount)/1e6, 2)  AS revenue_musd,
           ROUND(AVG(trip_distance), 2)     AS avg_miles,
           ROUND(100 * SUM(total_amount) / SUM(SUM(total_amount)) OVER (), 2) AS revenue_share_pct
    FROM trips
    GROUP BY pu_borough, pu_zone
    ORDER BY revenue_musd DESC
    LIMIT 10
''')
top_zones.show(truncate=False)

+----------+----------------------------+-------+------------+---------+-----------------+
|pu_borough|pu_zone                     |trips  |revenue_musd|avg_miles|revenue_share_pct|
+----------+----------------------------+-------+------------+---------+-----------------+
|Queens    |JFK Airport                 |3704539|304.85      |16.15    |14.86            |
|Queens    |LaGuardia Airport           |2499164|169.6       |9.73     |8.27             |
|Manhattan |Midtown Center              |3405246|86.22       |2.44     |4.2              |
|Manhattan |Upper East Side South       |3460753|71.51       |1.78     |3.49             |
|Manhattan |Times Sq/Theatre District   |2440494|70.7        |3.22     |3.45             |
|Manhattan |Upper East Side North       |3050584|64.34       |1.95     |3.14             |
|Manhattan |Penn Station/Madison Sq West|2509572|63.92       |2.37     |3.12             |
|Manhattan |Midtown East                |2594765|63.7        |2.36     |3.1              |

In [9]:
# SQL 2: tipping behaviour by payment type and time of day (cash tips are not recorded by TLC)
tips = spark.sql('''
    SELECT payment,
           CASE WHEN pickup_hour BETWEEN 6 AND 11 THEN '1 morning'
                WHEN pickup_hour BETWEEN 12 AND 16 THEN '2 afternoon'
                WHEN pickup_hour BETWEEN 17 AND 21 THEN '3 evening'
                ELSE '4 night' END                       AS daypart,
           COUNT(*)                                     AS trips,
           ROUND(AVG(tip_pct), 2)                       AS avg_tip_pct,
           ROUND(AVG(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) * 100, 1) AS pct_trips_tipped
    FROM trips
    WHERE payment IN ('Credit card', 'Cash')
    GROUP BY payment, daypart
    ORDER BY payment, daypart
''')
tips.show(truncate=False)

+-----------+-----------+--------+-----------+----------------+
|payment    |daypart    |trips   |avg_tip_pct|pct_trips_tipped|
+-----------+-----------+--------+-----------+----------------+
|Cash       |1 morning  |2590358 |0.0        |0.0             |
|Cash       |2 afternoon|3865052 |0.0        |0.0             |
|Cash       |3 evening  |3159900 |0.0        |0.0             |
|Cash       |4 night    |1602049 |0.0        |0.0             |
|Credit card|1 morning  |12593085|24.35      |94.2            |
|Credit card|2 afternoon|17081321|24.66      |95.1            |
|Credit card|3 evening  |19110900|26.36      |96.1            |
|Credit card|4 night    |10099454|25.56      |93.8            |
+-----------+-----------+--------+-----------+----------------+



In [10]:
# SQL 3: window function — busiest airport-bound route each month
routes = spark.sql('''
    WITH r AS (
        SELECT year_month, pu_zone, do_zone, COUNT(*) AS trips
        FROM trips
        WHERE do_zone IN ('JFK Airport', 'LaGuardia Airport', 'Newark Airport')
        GROUP BY year_month, pu_zone, do_zone
    )
    SELECT year_month, pu_zone, do_zone, trips
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY year_month ORDER BY trips DESC) AS rk FROM r)
    WHERE rk = 1
    ORDER BY year_month
''')
routes.show(24, truncate=False)

+----------+-------------------------+-----------------+-----+
|year_month|pu_zone                  |do_zone          |trips|
+----------+-------------------------+-----------------+-----+
|2023-01   |Times Sq/Theatre District|LaGuardia Airport|3139 |
|2023-02   |Times Sq/Theatre District|LaGuardia Airport|3027 |
|2023-03   |Times Sq/Theatre District|LaGuardia Airport|4274 |
|2023-04   |Times Sq/Theatre District|LaGuardia Airport|4006 |
|2023-05   |Times Sq/Theatre District|LaGuardia Airport|4373 |
|2023-06   |Times Sq/Theatre District|LaGuardia Airport|4469 |
|2023-07   |Times Sq/Theatre District|LaGuardia Airport|3818 |
|2023-08   |Times Sq/Theatre District|LaGuardia Airport|4015 |
|2023-09   |Times Sq/Theatre District|LaGuardia Airport|3760 |
|2023-10   |Times Sq/Theatre District|LaGuardia Airport|5504 |
|2023-11   |Times Sq/Theatre District|LaGuardia Airport|4641 |
|2023-12   |Times Sq/Theatre District|LaGuardia Airport|5259 |
|2024-01   |Times Sq/Theatre District|LaGuardia Airport

## 5. Execution plans & optimization
`explain("formatted")` prints the physical plan. Things to look for:
* **`PushedFilters`** in each `Scan parquet` node → predicates evaluated inside the Parquet reader (row-group skipping via min/max stats).
* **`ReadSchema`** containing only 10 columns → column pruning.
* **`BroadcastHashJoin`** (not `SortMergeJoin`) → no shuffle for the zone join.
* **`HashAggregate` → `Exchange hashpartitioning` → `HashAggregate`** → partial (map-side) aggregation before the shuffle.
* **`AdaptiveSparkPlan`** → AQE re-optimizes at runtime (coalesces shuffle partitions).

In [11]:
borough_hour.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (91)
+- Sort (90)
   +- Exchange (89)
      +- ObjectHashAggregate (88)
         +- Exchange (87)
            +- ObjectHashAggregate (86)
               +- Project (85)
                  +- BroadcastHashJoin LeftOuter BuildRight (84)
                     :- Project (79)
                     :  +- BroadcastHashJoin LeftOuter BuildRight (78)
                     :     :- Union (73)
                     :     :  :- Project (3)
                     :     :  :  +- Filter (2)
                     :     :  :     +- Scan parquet  (1)
                     :     :  :- Project (6)
                     :     :  :  +- Filter (5)
                     :     :  :     +- Scan parquet  (4)
                     :     :  :- Project (9)
                     :     :  :  +- Filter (8)
                     :     :  :     +- Scan parquet  (7)
                     :     :  :- Project (12)
                     :     :  :  +- Filter (11)
                     :     :  :     +-

In [12]:
top_zones.explain()   # SQL query: same Catalyst optimizer, same kind of physical plan

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=10, orderBy=[revenue_musd#2629 DESC NULLS LAST], output=[pu_borough#1824,pu_zone#1825,trips#2628L,revenue_musd#2629,avg_miles#2630,revenue_share_pct#2631])
   +- Project [pu_borough#1824, pu_zone#1825, trips#2628L, revenue_musd#2629, avg_miles#2630, round(((100.0 * _w0#2638) / _we0#2639), 2) AS revenue_share_pct#2631]
      +- Window [sum(_w0#2638) windowspecdefinition(specifiedwindowframe(RowFrame, unboundedpreceding$(), unboundedfollowing$())) AS _we0#2639]
         +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=8751]
            +- HashAggregate(keys=[pu_borough#1824, pu_zone#1825], functions=[count(1), sum(total_amount#45), avg(trip_distance#39)])
               +- Exchange hashpartitioning(pu_borough#1824, pu_zone#1825, 64), ENSURE_REQUIREMENTS, [plan_id=8748]
                  +- HashAggregate(keys=[pu_borough#1824, pu_zone#1825], functions=[partial_count(1), partial_sum(total_amoun

In [13]:
# Comparison: a *badly ordered* pipeline — join (without broadcast hint and with broadcasting disabled)
# BEFORE filtering. Catalyst still pushes simple filters below the join, but the join strategy
# becomes SortMergeJoin, which shuffles the whole trips table.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
bad = (trips_raw.join(pu, "PULocationID")
       .withColumn("duration_min",
                   (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
       .filter((F.col("duration_min") > 1) & (F.col("trip_distance") > 0))
       .groupBy("pu_borough").agg(F.count("*").alias("trips")))
bad.explain()
t = time.time(); bad.collect(); t_bad = time.time() - t
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

good = (trips_raw.filter((F.col("trip_distance") > 0))
        .withColumn("duration_min",
                    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .filter(F.col("duration_min") > 1)
        .join(F.broadcast(pu), "PULocationID")
        .groupBy("pu_borough").agg(F.count("*").alias("trips")))
t = time.time(); good.collect(); t_good = time.time() - t
print(f"SortMergeJoin version: {t_bad:.1f}s   |   filter-first + broadcast version: {t_good:.1f}s")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pu_borough#1824], functions=[count(1)])
   +- Exchange hashpartitioning(pu_borough#1824, 64), ENSURE_REQUIREMENTS, [plan_id=8983]
      +- HashAggregate(keys=[pu_borough#1824], functions=[partial_count(1)])
         +- Project [pu_borough#1824]
            +- SortMergeJoin [PULocationID#40], [PULocationID#1823], Inner
               :- Sort [PULocationID#40 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(PULocationID#40, 64), ENSURE_REQUIREMENTS, [plan_id=8975]
               :     +- Union
               :        :- Project [cast(PULocationID#7L as int) AS PULocationID#40]
               :        :  +- Filter (((isnotnull(trip_distance#4) AND ((cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(America/New_York), false) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(America/New_York), false)) as double) / 60.0) > 1.0)) AND (trip_distance#4 >

SortMergeJoin version: 41.9s   |   filter-first + broadcast version: 12.9s


## 6. Caching benchmark (bonus)

In [14]:
def run_queries(df):
    t = time.time()
    df.groupBy("pu_borough").count().collect()
    df.groupBy("payment").agg(F.avg("tip_pct")).collect()
    df.groupBy("pickup_hour").agg(F.avg("speed_mph")).collect()
    return time.time() - t

slim = enriched.select("pu_borough", "payment", "tip_pct", "pickup_hour", "speed_mph")
t_nocache = run_queries(slim)

slim.cache()
t = time.time(); slim.count(); t_materialize = time.time() - t   # first action fills the cache
t_cache = run_queries(slim)
print(f"3 queries without cache: {t_nocache:.1f}s")
print(f"cache materialization  : {t_materialize:.1f}s (one-time)")
print(f"3 queries with cache   : {t_cache:.1f}s  -> {t_nocache/t_cache:.1f}x faster")
slim.groupBy("pu_borough").count().explain()   # shows InMemoryTableScan instead of Scan parquet
slim.unpersist()

3 queries without cache: 109.9s
cache materialization  : 73.5s (one-time)
3 queries with cache   : 19.4s  -> 5.7x faster


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pu_borough#1824], functions=[count(1)])
   +- Exchange hashpartitioning(pu_borough#1824, 64), ENSURE_REQUIREMENTS, [plan_id=15213]
      +- HashAggregate(keys=[pu_borough#1824], functions=[partial_count(1)])
         +- InMemoryTableScan [pu_borough#1824]
               +- InMemoryRelation [pu_borough#1824, payment#1788, tip_pct#1772, pickup_hour#1730, speed_mph#1805], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           *(27) Project [pu_borough#1824, payment#1788, tip_pct#1772, pickup_hour#1730, speed_mph#1805]
                           +- *(27) BroadcastHashJoin [DOLocationID#41], [DOLocationID#1829], LeftOuter, BuildRight, false
                              :- *(27) Project [DOLocationID#41, pickup_hour#1730, tip_pct#1772, payment#1788, speed_mph#1805, pu_borough#1824]
     

DataFrame[pu_borough: string, payment: string, tip_pct: double, pickup_hour: int, speed_mph: double]

## 7. Write results

In [15]:
OUT = "output"
t = time.time()
# Partition the cleaned fact table by month: downstream queries filtering on year_month only read
# the matching directories (partition pruning). repartition by the same key -> ~1 file per partition.
(enriched.repartition("year_month")
    .write.mode("overwrite").partitionBy("year_month")
    .parquet(f"{OUT}/trips_clean"))
borough_hour.coalesce(1).write.mode("overwrite").parquet(f"{OUT}/borough_hour")
top_zones.coalesce(1).write.mode("overwrite").parquet(f"{OUT}/top_zones")
tips.coalesce(1).write.mode("overwrite").parquet(f"{OUT}/tips_by_payment_daypart")
routes.coalesce(1).write.mode("overwrite").parquet(f"{OUT}/top_airport_route_by_month")
monthly.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUT}/monthly_summary_csv")
print(f"written in {time.time()-t:.1f}s")

# verify + show partition pruning on read-back
back = spark.read.parquet(f"{OUT}/trips_clean").filter(F.col("year_month") == "2024-06")
print("rows in 2024-06:", back.count())
back.explain()   # look for PartitionFilters: [isnotnull(year_month), (year_month = 2024-06)]

written in 1269.7s


rows in 2024-06: 2994458
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [DOLocationID#4881,PULocationID#4882,tpep_pickup_datetime#4883,tpep_dropoff_datetime#4884,passenger_count#4885,trip_distance#4886,payment_type#4887,fare_amount#4888,tip_amount#4889,total_amount#4890,duration_min#4891,pickup_hour#4892,pickup_dow#4893,tip_pct#4894,payment#4895,speed_mph#4896,pu_borough#4897,pu_zone#4898,do_borough#4899,do_zone#4900,year_month#4901] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/zhaoxuechun/Library/Application Support/Claude/scratch-wor..., PartitionFilters: [isnotnull(year_month#4901), (year_month#4901 = 2024-06)], PushedFilters: [], ReadSchema: struct<DOLocationID:int,PULocationID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime...




## 8. MLlib (optional) — predict fare from trip features

In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ml_df = (enriched.filter(F.col("year_month") == "2024-06")      # one month is plenty for a demo
         .select("fare_amount", "trip_distance", "duration_min", "pickup_hour", "pickup_dow",
                 "passenger_count", F.coalesce("pu_borough", F.lit("Unknown")).alias("pu_borough"))
         .sample(0.2, seed=42))
train, test = ml_df.randomSplit([0.8, 0.2], seed=42)

pipe = Pipeline(stages=[
    StringIndexer(inputCol="pu_borough", outputCol="borough_idx", handleInvalid="keep"),
    OneHotEncoder(inputCols=["borough_idx"], outputCols=["borough_vec"]),
    VectorAssembler(inputCols=["trip_distance", "duration_min", "pickup_hour", "pickup_dow",
                               "passenger_count", "borough_vec"], outputCol="features"),
    GBTRegressor(featuresCol="features", labelCol="fare_amount", maxIter=30, maxDepth=5, seed=42),
])
t = time.time()
model = pipe.fit(train)
pred = model.transform(test)
ev = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")
print(f"train rows {train.count():,} | fit {time.time()-t:.1f}s")
print(f"RMSE = {ev.evaluate(pred, {ev.metricName: 'rmse'}):.2f} USD,  R2 = {ev.evaluate(pred, {ev.metricName: 'r2'}):.3f}")
fi = model.stages[-1].featureImportances
names = ["trip_distance", "duration_min", "pickup_hour", "pickup_dow", "passenger_count"]
print({n: round(float(fi[i]), 3) for i, n in enumerate(names)})

train rows 480,326 | fit 584.7s


RMSE = 5.87 USD,  R2 = 0.897
{'trip_distance': 0.785, 'duration_min': 0.155, 'pickup_hour': 0.018, 'pickup_dow': 0.006, 'passenger_count': 0.004}


In [17]:
# Keep the session alive while taking Spark UI screenshots (http://localhost:4040), then stop it.
# spark.stop()